# 01. Download raw data (PRIDE PXD029845)

Kim et al. 2022, *Transl Psychiatry* 12:142; prefrontal cortex (BA9/10), TMT 10-plex, 23 suicide decedents and 13 sudden-death controls.

The file list, sizes and SHA-1 checksums come from the PRIDE API and are stored in `config/PXD029845_manifest.tsv`.
Files that are already present are not downloaded again, and checksums that were already verified are read from `results/sha1_verified.json`, so the notebook can be re-run at any time and only does what is missing.

In [1]:
import hashlib, json, re, shutil, time
from pathlib import Path
import pandas as pd
import requests
from tqdm import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data_raw"          # change here to place raw files on another disk
RAW.mkdir(exist_ok=True)
LOG = ROOT / "results" / "sha1_verified.json"

manifest = pd.read_csv(ROOT / "config" / "PXD029845_manifest.tsv", sep="\t")
manifest.groupby("category")["size_bytes"].agg(files="count", gb=lambda s: round(s.sum() / 1e9, 1))

,files,gb
category,,
FASTA,1,0.0
OTHER,2,2.0
RAW,96,200.1
SEARCH,1,86.9


## What is needed

The 96 raw files, the FASTA file used by the authors and their protein table. The Proteome Discoverer result files (`.msf`, 87 GB; `.pep.xml`, 2 GB) are not needed.

In [2]:
wanted = manifest[(manifest["category"] == "RAW") | manifest["file_name"].str.lower().str.endswith((".fasta", ".txt"))].reset_index(drop=True)

def locate(name):
    """Path of a file in data_raw or in one of its TMT-set folders (None if absent)."""
    hits = list(RAW.rglob(name))
    return hits[0] if hits else None

wanted["present"] = [(p := locate(n)) is not None and p.stat().st_size == s for n, s in zip(wanted["file_name"], wanted["size_bytes"])]
print(len(wanted), "files,", round(wanted["size_bytes"].sum() / 1e9, 1), "GB;", int(wanted["present"].sum()), "already present with the expected size")

98 files, 200.2 GB; 98 already present with the expected size


## Download what is missing

Transfers resume after interruption. `raise_for_status` makes sure that an error page returned by the server is never written into a data file (this happened with a simpler downloader and is repaired by `tools/repair_html_inserts.py`).

In [3]:
def download(row, retries=20):
    target = locate(row.file_name) or RAW / row.file_name
    for attempt in range(retries):
        have = target.stat().st_size if target.exists() else 0
        if have == row.size_bytes:
            return True
        if have > row.size_bytes:
            target.unlink(); have = 0
        try:
            with requests.get(row.url, headers={"Range": f"bytes={have}-"}, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(target, "ab") as fh, tqdm(total=row.size_bytes, initial=have, unit="B", unit_scale=True, desc=row.file_name, leave=False) as bar:
                    for chunk in r.iter_content(1 << 20):
                        fh.write(chunk); bar.update(len(chunk))
        except requests.RequestException as exc:
            print(row.file_name, "retry", attempt + 1, type(exc).__name__); time.sleep(15)
    return False

missing = wanted[~wanted["present"]]
for row in tqdm(list(missing.itertuples()), desc="downloading"):
    download(row)
print("downloaded in this run:", len(missing))

downloading: 0it [00:00, ?it/s]

downloaded in this run: 0


## Verify checksums

In [4]:
def sha1_of(path, block=1 << 24):
    h = hashlib.sha1()
    with open(path, "rb") as fh:
        while chunk := fh.read(block):
            h.update(chunk)
    return h.hexdigest()

verified = json.loads(LOG.read_text()) if LOG.exists() else {}
for row in tqdm(list(wanted.itertuples()), desc="verifying"):
    if verified.get(row.file_name) is True:
        continue
    path = locate(row.file_name)
    verified[row.file_name] = bool(path is not None and path.stat().st_size == row.size_bytes and sha1_of(path) == row.sha1)
    LOG.write_text(json.dumps(verified, indent=1))
status = pd.Series({n: verified.get(n, False) for n in wanted["file_name"]})
print(f"{int(status.sum())} of {len(status)} files match the PRIDE SHA-1 checksum")
assert status.all(), list(status[~status].index)

verifying: 100%|██████████| 98/98 [00:00<?, ?it/s]

98 of 98 files match the PRIDE SHA-1 checksum


## Organise by TMT set

FragPipe needs the files of each TMT set (plex) in their own folder, next to the channel annotation file written by notebook 04.

In [5]:
for f in sorted(RAW.glob("*.raw")):
    s = re.search(r"SET_(\d)_frxn", f.name).group(1)
    (RAW / f"SET_{s}").mkdir(exist_ok=True)
    shutil.move(str(f), RAW / f"SET_{s}" / f.name)
{d.name: len(list(d.glob("*.raw"))) for d in sorted(RAW.glob("SET_*"))}

{'SET_1': 24, 'SET_2': 24, 'SET_3': 24, 'SET_4': 24}